# QTL-Level Modeling of Phenotype Using PLSR

This notebook investigates the relationship between genetic variation (QTL/loci) and phenotypic traits (e.g., VOC profiles) using Partial Least Squares Regression (PLSR).

Genotype and phenotype datasets are first aligned and preprocessed to ensure consistency across samples. A PLSR model is then trained to identify latent components that capture the maximal covariance between genetic markers and phenotypic traits.

The analysis aims to:

* Predict phenotypic variation from genetic data
* Identify key loci contributing to trait variation
* Quantify how well genetic information explains phenotype

This approach provides a predictive framework for linking genetic architecture to metabolomic traits.


# Loading plsr module

In [0]:
import os, sys, importlib
import yaml
import inspect

In [0]:


MODULE_PATH = "/Volumes/bmqg/default_bronze/fatemeh/final_project/modules"
assert os.path.exists(f"{MODULE_PATH}/PLSR.py"), "PLSR.py not found at this path!"

if MODULE_PATH not in sys.path:
    sys.path.insert(0, MODULE_PATH)




import PLSR as plsr_mod
importlib.reload(plsr_mod)


<module 'PLSR' from '/Volumes/bmqg/default_bronze/fatemeh/final_project/modules/PLSR.py'>

In [0]:

CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"
with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

CONFIG["params"] = CONFIG["parameters"]
CONFIG["paths"]["plsr_out_dir"] = "/Volumes/bmqg/default_bronze/fatemeh/final_project/plsr"


print(inspect.getsource(plsr_mod.run_fast_plsr_pipeline))
os.makedirs(CONFIG["paths"]["plsr_out_dir"], exist_ok=True)

print("Using GWAS table:", CONFIG["data"]["gwas_table"])

res = plsr_mod.run_fast_plsr_pipeline(
    spark=spark,
    config=CONFIG,
    drop_ch00=True,
    make_plots=True
)

display(res["all_top"].head())
print(res["out_csv"])

def run_fast_plsr_pipeline(spark, config, drop_ch00=True, make_plots=False):
    gwas_table = config["data"]["gwas_table"]

    params = config.get("params", config.get("parameters", {}))
    distance = params.get("distance_threshold", 250_000)
    max_qtls = params.get("max_qtls_for_pivot", 300)
    n_comp = params.get("n_components", 2)
    top_n = params.get("top_n", 10)

    out_dir = config["paths"]["plsr_out_dir"]
    os.makedirs(out_dir, exist_ok=True)

    print(f"[run_fast_plsr_pipeline] Using GWAS table: {gwas_table}")

    gwas_df = spark.table(gwas_table).cache()
    gwas_df.count()  # materialize cache

    traits = (
        gwas_df.select("trait")
        .distinct()
        .toPandas()["trait"]
        .astype(str)
        .tolist()
    )

    marker_gqtl, gqtl_bounds = build_global_qtl_blocks(
        spark, traits, gwas_table, distance, drop_ch00
    )

    marker_gqtl = marker_gqtl.cache()
    gqtl_bounds = gqtl_bounds.cache()
    marker_gqtl.count()
    gqtl_bounds.

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
import pandas as pd

# load file
df = pd.read_csv("/Volumes/bmqg/default_bronze/fatemeh/final_project/plsr/plsr_top_loadings_fast.csv")

# extract chromosome from QTL_id
df["chrom"] = df["QTL_id"].str.extract(r'ch(\d+)')

# optional: convert to int for sorting
df["chrom"] = df["chrom"].astype(int)

# =========================
# 1) importance per chromosome
# =========================
chrom_importance = (
    df.groupby("chrom")["abs"]
      .sum()
      .sort_values(ascending=False)
)

print("Chromosome importance (sum of abs loadings):")
print(chrom_importance)

# =========================
# 2) number of QTLs per chromosome
# =========================
chrom_counts = df["chrom"].value_counts().sort_index()

print("\nQTL count per chromosome:")
print(chrom_counts)

# =========================
# 3) combine both
# =========================
summary = pd.DataFrame({
    "total_importance": chrom_importance,
    "n_QTLs": chrom_counts
}).sort_values("total_importance", ascending=False)

print("\nFinal summary:")
print(summary)

Chromosome importance (sum of abs loadings):
chrom
12    48.077149
7     26.939610
1     25.123292
5     21.456658
2     10.202893
4      9.036195
11     8.773417
8      6.415619
6      4.098283
3      2.070061
10     1.834829
9      1.026745
Name: abs, dtype: float64

QTL count per chromosome:
chrom
1     106
2      40
3       7
4      38
5      88
6      18
7     107
8      27
9       4
10      7
11     35
12    193
Name: count, dtype: int64

Final summary:
       total_importance  n_QTLs
chrom                          
12            48.077149     193
7             26.939610     107
1             25.123292     106
5             21.456658      88
2             10.202893      40
4              9.036195      38
11             8.773417      35
8              6.415619      27
6              4.098283      18
3              2.070061       7
10             1.834829       7
9              1.026745       4


Chromosome 12 is the dominant and amplified hotspot controlling VOC variation in old potato, with additional contributions from chromosomes 7, 1, and 5.